## Introduction to NLP Interpretability

The field of Natural Language Processing (NLP) interpretability aims to understand the internal mechanisms of "black-box" models like Large Language Models (LLMs). Understanding these mechanisms is crucial for deploying safe, trustable, and unbiased systems in real-world settings.

The interpretability community gained significant traction around 2016 with analyses of Recurrent Neural Networks (LSTMs). The introduction of Transformers in 2017—a highly effective but opaque architecture—sparked a boom in this field, leading to dedicated tracks and the first Blackbox NLP workshop at the ACL conference in 2018.

Early techniques included:

* **Distributional Semantics:** Analyzing vector space analogies (e.g., Word2Vec's $\text{King} - \text{Man} + \text{Woman} = \text{Queen}$).
* **Attention Maps:** Visualizing which tokens receive the highest attention weights.
* **Neuron Analysis:** Analyzing individual neurons responsible for specific outputs (borrowed from computer vision saliency maps).

---

## Taxonomy of Interpretability Techniques

Interpretability techniques broadly fall into two main categories:

| Category | Description | Primary Techniques |
| --- | --- | --- |
| **Behavior Localization** | Identifying and localizing how specific model components or input tokens contribute to a particular output generation. | Input attribution, Causal interventions, Circuit analysis. |
| **Information Decoding** | Decoding and learning what information is stored within the model's internal representations (hidden states/activations). | Probing, Decoding in vocabulary space (Logit Lens), Dictionary learning. |

---

## Information Decoding: Probing

**Probing** is a foundational technique to understand what linguistic information a specific layer of an LLM contains.

**Mechanism:**
Given an $n$-layered transformer, freeze the model's weights. Train an external classifier $G$ (the probe) on top of the hidden state $H$ of the $L$-th layer to predict a specific linguistic feature $Z$ (e.g., Part-of-Speech (POS) tags, coreference resolution).

**Information Theory Perspective:**
A probe attempts to model the **Mutual Information** $I(H; Z)$ between the hidden state $H$ and the linguistic property $Z$. It measures how much information $H$ can tell us about $Z$.

### The Challenge of Probe Complexity

If a simple probe (e.g., 2-layer MLP) fails but a complex probe (e.g., 15-layer MLP) succeeds, it becomes ambiguous whether the linguistic information was actually encoded in the model's hidden states, or if the complex probe merely memorized the dataset.

**Solution: Control Tasks & Selectivity**
To test if a probe is memorizing, researchers design a **Control Task**.

1. Randomly assign labels to tokens (e.g., randomize 40 POS tags across the text). This removes all linguistic structure.
2. Train the probe on the true task and the control task.
3. Measure **Selectivity**: The difference in accuracy between the probe task and the control task.

> **Key Takeaway:** An effective probe should have high accuracy on the actual linguistic task and low accuracy on the randomized control task (yielding high selectivity). If control task accuracy spikes as probe complexity increases, the probe is simply memorizing.

---

## Mechanistic Interpretability (MI)

Emerging around 2020, Mechanistic Interpretability attempts to reverse-engineer the algorithms implemented by neural networks into human-understandable mechanisms. Borrowing from natural sciences (studying atoms to understand materials), MI studies the "atoms" of neural networks: **neurons and features**.

### Core Concepts & Terminology

* **Features:** The fundamental units of neural networks (analogous to cells).
* **Circuits:** Computational subgraphs. Nodes represent neurons/activations, and edges represent the weights transferring activations. The hypothesis is that only a specific subgraph of neurons contributes to a specific task.
* **Universality:** The claim that analogous features and circuits form across different models and tasks (though generalizability remains highly debated).
* **Polysemanticity:** The phenomenon where a single neuron encodes multiple, unrelated properties (e.g., a "car" neuron activating for windows, wheels, and car bodies simultaneously). Polysemanticity prevents human understanding and is a major hurdle in MI.

### Three Steps to Circuit Discovery

1. **Isolate a Task and Define a Metric:** Create a dataset that reproduces a specific behavior. Example tasks include:
* *Indirect Object Identification (IOI):* "John and Mary went to the market. John gave a pen to [Mary]."
* *Docstrings / Induction:* Predicting next tokens based on previous contexts.
* *Metric:* Typically the logit difference between the correct target and an incorrect target (e.g., Logit(Mary) - Logit(John)).


2. **Define Scope of Interpretation:** Determine the granularity of the computational Directed Acyclic Graph (DAG) (e.g., full layers, attention heads, or fine-grained Query/Key/Value activations).
3. **Perform Activation Patching:** Systematically corrupt and replace components to isolate the critical subgraph.

---

## Activation Patching vs. Attribution Patching

To find which components belong to a circuit, we perform patching.

### 1. Activation Patching (ACDC Algorithm)

* **Process:**
1. Run a clean input ("The Eiffel Tower is in [Paris]") and cache the clean activations $a_c$.
2. Run a corrupted input ("The Colosseum is in [Rome]") to get corrupted activations $a_r$.
3. Iteratively overwrite the activation of a single node in the corrupted run with its clean activation $a_c$.
4. Observe the change in the output metric (logit difference or KL divergence). If the metric shifts significantly toward the clean output, the component is part of the circuit.


* **Drawback:** Computationally exhaustive. If there are $N$ components, it requires $N$ distinct forward passes.

### 2. Attribution Patching (Linear Approximation)

Attribution patching uses a first-order Taylor expansion to approximate the logit difference without running $N$ forward passes.

Let $f(a)$ be the function mapping an activation $a$ to the output logit difference. We want to approximate $f(a_c) - f(a_r)$. Using linear approximation around the corrupted activation $a_r$:

$$f(a_c) \approx f(a_r) + \left( \frac{\partial f}{\partial a} \bigg\vert{}_{a=a_r} \right) \cdot (a_c - a_r)$$

Rearranging yields the change in logit difference:

$$f(a_c) - f(a_r) \approx \left( \frac{\partial f}{\partial a} \bigg\vert{}_{a=a_r} \right) \cdot (a_c - a_r)$$

* **Computational Advantage:** Requires only **two forward passes** (clean and corrupted) to get $a_c$ and $a_r$, and **one single backward pass** (backpropagation) on the corrupted run to obtain the gradient $\frac{\partial f}{\partial a}$ for all activations simultaneously.
* **Assumptions/Limitations:** Valid only if the difference $(a_c - a_r)$ is very small. Therefore, it only works when patching single tokens at fine granularity (like individual attention heads). It breaks down for large perturbations like entire layers.

---

## Notable Circuit Findings

### Induction Heads (In-Context Learning)

Induction heads are two-head circuits responsible for pattern-copying (in-context learning).

* **Head 1:** Looks back at previous tokens to find the current context (e.g., finds a previous instance of token "A").
* **Head 2:** Identifies the token that immediately followed the previous instance (e.g., "B") and copies it to predict the next token.
* **Emergence:** In-context learning emerges suddenly during training. This is measured by the metric $(L_{50} - L_{500})$, which is the loss at the 50th token minus the loss at the 500th token. A sudden drop in this metric perfectly coincides with the formation of induction heads. Single-layer models cannot form induction heads and do not exhibit this drop.

### Chain of Thought (CoT) Reasoning

Mechanistic analysis reveals that CoT relies on three subtasks, exhibiting a **functional rift**:

1. **Decision making:** Selecting the entity.
2. **Copying:** Fetching information from the context (similar to induction).
3. **Inductive reasoning:** Using rules.
*The first half of the model's layers primarily moves information through the residual stream, while the second half computes the actual answers.*

---

## Advanced Decoding Techniques

### Decoding in Vocabulary Space (Logit Lens)

Instead of waiting for the final layer, Logit Lens projects the intermediate hidden state $h_l$ of layer $l$ directly into the vocabulary space using the model's unembedding matrix $W_U$:

$$\text{Vocabulary Distribution} = W_U \times h_l$$

This allows researchers to see which tokens are being "pushed" or promoted at early and middle layers, tracking how the output concept forms during the forward pass. (Also used in Vision-Language models to see what pixels correspond to which vocabulary tokens).

### Path Scopes

Path scopes generalize the probing framework by mapping activations between two different models ($M$ and $M'$).

* **Example Use:** Extracting an activation corresponding to the entity "India" from Model $M$'s processing of "The largest river in India", and patching it into Model $M'$ processing "The largest city of X is". If Model $M'$ outputs the correct city for India, it proves Model $M$'s activation stored generalized entity information.
* If $M = M'$ and the activation is patched at the very last layer, Path Scopes becomes mathematically equivalent to Logit Lens.

---

## Dictionary Learning & Sparse Autoencoders (SAEs)

### The Superposition Hypothesis

Models often need to embed more features than they have dimensions in their latent space (e.g., fitting 1,000 features into a 512-dimensional vector space). Consequently, features cannot align perfectly with orthogonal basis vectors. They are stored in **superposition**, causing polysemanticity.

* **Privileged Basis:** Some architectures (like Transformer MLPs) explicitly incentivize features to align with the basis vectors, making them somewhat easier to disentangle.

### Sparse Autoencoders

Dictionary learning uses Sparse Autoencoders (SAEs) to map dense, polysemantic activations $x$ into a much higher-dimensional space where features can be separated (monosemanticity).

* **Encoder:** Projects $x$ (e.g., 512-dim) into a highly expanded space $f_i(x)$ (e.g., 131,000-dim) using an encoder matrix $W_{enc}$.
* **Decoder:** Attempts to reconstruct $x'$ using a decoder matrix.
* **Training Objective:** Unsupervised training minimizing reconstruction loss, with a heavy **$L_1$ penalty** applied to the activations $f_i$. The $L_1$ norm forces sparsity, pushing most components of $f_i$ to zero.

**Real-world Impact:** In models like Claude 3 Sonnet, SAEs have successfully extracted millions of disentangled, human-interpretable features. Researchers isolated a specific feature vector responsible for "gender bias," allowing them to manually intervene, scale down the feature, and successfully control the bias in the model's output.

---

## Practice Questions

**Q1. In the context of probing, a researcher finds that a 2-layer MLP probe achieves 60% accuracy on POS tagging, while a 15-layer MLP achieves 95%. However, on a control task with randomized labels, the 2-layer MLP achieves 10% and the 15-layer MLP achieves 90%. What can be concluded about the LLM's representations?**
A) The LLM encodes POS tagging information non-linearly, requiring a deep probe.
B) The 15-layer probe is exhibiting high selectivity and successfully decoding the hidden state.
C) The 15-layer probe is memorizing the mappings, and there is little evidence the LLM inherently encodes the POS information.
D) The mutual information between the hidden state and the linguistic property increases with probe depth.
*Answer: C. Explanation: High control task accuracy indicates the probe is complex enough to memorize randomized data. Selectivity (Probe Accuracy - Control Accuracy) for the 15-layer MLP is only 5%, meaning the probe is doing the heavy lifting, not the model's embeddings.*

**Q2. Which of the following is the primary advantage of Attribution Patching over standard Activation Patching (ACDC)?**
A) It avoids the polysemanticity problem entirely.
B) It requires only two forward passes and one backward pass to approximate component importance.
C) It allows for the accurate patching of entire layers and multi-token sequences.
D) It projects the hidden states directly into the vocabulary space without backpropagation.
*Answer: B. Explanation: Activation patching requires $N$ forward passes for $N$ components. Attribution patching uses linear approximation (Taylor expansion) requiring only the gradients from a single corrupted backward pass. Option C is false because linear approximation fails for large structural changes like full layers.*

**Q3. The emergence of Induction Heads in a Transformer model directly correlates with which observable metric?**
A) High selectivity on control tasks.
B) A sudden drop in the reconstruction loss of a Sparse Autoencoder.
C) A sharp decrease in the difference between the loss at the 50th token and the 500th token ($L_{50} - L_{500}$).
D) The alignment of features with a privileged basis.
*Answer: C. Explanation: The transcript explicitly ties the emergence of induction heads to the $L_{50} - L_{500}$ in-context learning metric dropping towards zero, meaning the model utilizes deep context just as well as early context.*

**Q4. According to the linear representation and superposition hypotheses, why do neurons exhibit polysemanticity?**
A) Because MLPs naturally reject privileged bases.
B) Because the model attempts to embed more features than the dimensions available in its latent subspace.
C) Because the vocabulary unembedding matrix $W_U$ forces compression.
D) Because Sparse Autoencoders force the $L_1$ norm to be zero.
*Answer: B. Explanation: Superposition occurs when the number of features exceeds the available dimensions, forcing multiple features to share the same neuronal representations, leading to polysemanticity.*

---

## Revision Summary

* **Interpretability Types:** Divided into Behavior Localization (finding *where* things happen via patching) and Information Decoding (finding *what* is stored via probing/logit lens).
* **Probing:** Uses classifiers to extract linguistic info from hidden states. Evaluated using **Control Tasks** to test **Selectivity** (preventing probe memorization).
* **Mechanistic Interpretability:** Reverses-engineers models into features and circuits. Fights **Polysemanticity** (one neuron, many meanings).
* **Patching:**
* *Activation Patching:* Replaces activations to track logit differences. Highly accurate but computationally expensive.
* *Attribution Patching:* Uses Taylor expansion for fast linear approximation (2 forward + 1 backward pass). Good for fine-grained, single-token analysis.


* **Key Circuits:** **Induction Heads** drive in-context learning. **CoT** shows a functional rift (first half moves info, second half calculates).
* **Decoding:** **Logit Lens** ($W_U \times h_l$) projects intermediate layers to vocab space. **Path Scopes** generalises this across different models.
* **Sparse Autoencoders (SAEs):** Solves superposition by mapping dense polysemantic activations into a massive, sparse, high-dimensional space (using an $L_1$ penalty) to achieve **monosemantic**, human-interpretable features.